# Complete Guide to Selective Prediction

This notebook demonstrates all selective prediction methods in `incerto` — models that
know when to abstain from making predictions.

**What you'll learn:**
- What selective prediction is and when to use it
- 4 methods: SoftmaxThreshold, DeepGambler, SelectiveNet, Self-Adaptive Training
- Custom loss functions: gambler_loss, selective_loss, sat_loss
- Metrics: coverage, risk, AURC, accuracy-coverage curves
- Risk-coverage visualization and method comparison

**Runtime:** ~5 min (MPS/CUDA), ~15 min (CPU)

## Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Selective prediction methods
from incerto.sp import (
    SoftmaxThreshold,
    DeepGambler,
    SelectiveNet,
    SelfAdaptiveTraining,
    make,
)

# Metrics
from incerto.sp import coverage, risk, aurc, accuracy_coverage_curve

# Visualization
from incerto.sp import plot_risk_coverage, plot_accuracy_coverage

from incerto.utils import ConvNet, seed_everything

seed_everything(42)

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## Part 1: What is Selective Prediction?

**Selective prediction** (prediction with rejection) allows models to abstain on
uncertain inputs, trading coverage for accuracy.

**Key concepts:**
- **Coverage:** Fraction of samples the model chooses to predict on
- **Selective risk:** Error rate on non-rejected samples
- **AURC:** Area under the risk-coverage curve (lower is better)

**When to use it:**
- Cost of error is high (medical, financial, autonomous systems)
- Human review is available for rejected samples
- Model confidence correlates with correctness

| Method | Approach | Training |
|--------|----------|----------|
| SoftmaxThreshold | Threshold max softmax probability | Post-hoc (no training) |
| DeepGambler | Extra abstain class + gambler's loss | End-to-end |
| SelectiveNet | Dedicated selection head g(x) | End-to-end |
| Self-Adaptive Training | Blend hard/soft labels | End-to-end |

## Part 2: Data and Standard Model

We use **Fashion-MNIST** — a harder drop-in replacement for MNIST with 10 clothing
categories. Unlike MNIST (99%+ accuracy), Fashion-MNIST (~91%) produces natural
uncertainty, making it ideal for demonstrating selective prediction.

In [ ]:
# Load Fashion-MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST('./data', train=False, transform=transform)

# Optimized data loaders
num_workers = min(4, os.cpu_count() or 0)
pin_memory = device.type == "cuda"
loader_kwargs = dict(num_workers=num_workers, pin_memory=pin_memory, persistent_workers=num_workers > 0)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, **loader_kwargs)

CLASS_NAMES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print(f"Training: {len(train_dataset)} | Test: {len(test_dataset)}")
print(f"DataLoader: num_workers={num_workers}, pin_memory={pin_memory}")

In [ ]:
# Train a standard CNN (baseline for SoftmaxThreshold and comparison)
model = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
print("Training standard model (10 epochs)...")
for epoch in range(10):
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Acc={100.*correct/total:.2f}%")

print("Done!")

In [ ]:
# Helper to collect logits and labels from a data loader
def collect_logits(mdl, loader):
    mdl.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            logits = mdl(inputs.to(device))
            all_logits.append(logits.cpu())
            all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)

test_logits, test_labels = collect_logits(model, test_loader)
base_preds = test_logits.argmax(dim=-1)
base_acc = (base_preds == test_labels).float().mean()

print(f"Standard model accuracy: {base_acc:.2%}")

## Part 3: Method 1 — SoftmaxThreshold (MSP)

The simplest approach: use the **maximum softmax probability** as confidence
and reject samples below a threshold. No additional training required.

```python
predictor = SoftmaxThreshold(model)
logits, confidence = predictor(x, return_confidence=True)
should_reject = predictor.reject(confidence, threshold=0.9)
```

In [ ]:
predictor = SoftmaxThreshold(model)
predictor.eval()

# Get confidence scores (max softmax probability)
msp_conf = F.softmax(test_logits, dim=-1).max(dim=-1).values

# Sweep thresholds to show the coverage-accuracy tradeoff
thresholds = [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]

print("SoftmaxThreshold -- Confidence Threshold Sweep:")
print(f"{'Threshold':<12} {'Coverage':>10} {'Sel. Acc':>10} {'Risk':>10}")
print("-" * 45)

for thresh in thresholds:
    rejected = predictor.reject(msp_conf, threshold=thresh)
    cov = coverage(rejected)
    r = risk(base_preds, test_labels, rejected)
    sel_acc = 1.0 - r
    print(f"{thresh:<12.2f} {cov:>10.1%} {sel_acc:>10.2%} {r:>10.4f}")

# Compute AURC
sorted_conf, sorted_idx = msp_conf.sort(descending=True)
sorted_errors = (base_preds[sorted_idx] != test_labels[sorted_idx]).float()
msp_aurc = aurc(sorted_conf, sorted_errors)
print(f"\nAURC: {msp_aurc:.4f}")

## Part 4: Method 2 — DeepGambler

DeepGambler adds an extra **abstain class** and trains with the **gambler's loss**:

$$L = -\log\left(p_y + \frac{r}{o}\right)$$

where $p_y$ is the probability of the true class, $r = P(\text{abstain})$, and $o$ is the
reward for correct prediction. Higher reward penalizes abstention, pushing towards prediction.

**Reward annealing** (recommended): Start with a large reward so the model first learns
good features, then decrease it to allow selective abstention.

*Reference: Ziyin et al., "Deep Gamblers", NeurIPS 2019.*

In [ ]:
seed_everything(42)

# Feature backbone: ConvNet without classification head (128-d output)
dg_backbone = ConvNet(num_classes=10, dropout_rate=0.0)
dg_backbone.fc2 = nn.Identity()  # Remove classifier -> 128-d features

gambler = DeepGambler(dg_backbone, num_classes=10, num_features=128).to(device)
optimizer = torch.optim.Adam(gambler.parameters(), lr=0.001)

# Reward annealing: start high (learn features first), decay to target.
# High reward → abstaining is costly → model must predict.
# Low reward  → abstaining is cheaper → model can be selective.
target_reward = 2.2
num_epochs = 10
reward_schedule = [max(target_reward, 128 * 0.5 ** epoch) for epoch in range(num_epochs)]

print(f"Training DeepGambler ({num_epochs} epochs, reward annealing)...")
gambler.train()
for epoch in range(num_epochs):
    reward = reward_schedule[epoch]
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = gambler(inputs)  # (batch, 11) = 10 classes + 1 abstain
        loss = DeepGambler.gambler_loss(logits, labels, reward=reward)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = logits[:, :-1].max(1)  # Ignore abstain logit
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Acc={100.*correct/total:.2f}%, reward={reward:.1f}")

# Evaluate
dg_logits, dg_labels = collect_logits(gambler, test_loader)
dg_class_logits = dg_logits[:, :-1]  # First 10 columns = class logits
dg_preds = dg_class_logits.argmax(dim=-1)
dg_conf = gambler.confidence_from_logits(dg_logits)  # 1 - P(abstain)

dg_acc = (dg_preds == dg_labels).float().mean()
sorted_conf, sorted_idx = dg_conf.sort(descending=True)
sorted_errors = (dg_preds[sorted_idx] != dg_labels[sorted_idx]).float()
dg_aurc = aurc(sorted_conf, sorted_errors)

print(f"\nDeepGambler: Acc={dg_acc:.2%}, AURC={dg_aurc:.4f}")
print(f"Mean P(abstain) = {(1 - dg_conf).mean():.4f}")

## Part 5: Method 3 — SelectiveNet

SelectiveNet adds a dedicated **selection head** $g(x)$ that outputs a selection
probability in $[0, 1]$. The model is trained with a combined loss:

$$L = L_{\text{selective}} + \lambda \cdot \max(0,\; c - \Phi)^2$$

where $L_{\text{selective}}$ is selection-weighted cross-entropy, $c$ is the
coverage target, $\Phi$ is empirical coverage, and $\lambda$ is the penalty weight.

*Reference: Geifman & El-Yaniv, "SelectiveNet", ICML 2019.*

In [ ]:
seed_everything(42)

sn_backbone = ConvNet(num_classes=10, dropout_rate=0.0)
sn_backbone.fc2 = nn.Identity()

selnet = SelectiveNet(
    sn_backbone, num_classes=10, num_features=128,
    alpha=0.7,   # Coverage target
    lam=32.0,    # Penalty weight for coverage constraint
).to(device)
optimizer = torch.optim.Adam(selnet.parameters(), lr=0.001)

print("Training SelectiveNet (10 epochs, target coverage=70%)...")
selnet.train()
for epoch in range(10):
    total_loss, correct, total = 0, 0, 0
    epoch_cov = 0
    n_batches = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, selection = selnet(inputs, return_confidence=True)
        loss = selnet.selective_loss(logits, labels, selection)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        epoch_cov += selection.mean().item()
        n_batches += 1
    if (epoch + 1) % 2 == 0:
        avg_cov = epoch_cov / n_batches
        print(f"  Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Acc={100.*correct/total:.2f}%, Cov={avg_cov:.2%}")

# Evaluate: need return_confidence=True for SelectiveNet's selection head
selnet.eval()
sn_logits_list, sn_conf_list, sn_labels_list = [], [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        logits, sel = selnet(inputs.to(device), return_confidence=True)
        sn_logits_list.append(logits.cpu())
        sn_conf_list.append(sel.cpu())
        sn_labels_list.append(labels)

sn_logits = torch.cat(sn_logits_list)
sn_conf = torch.cat(sn_conf_list)
sn_labels = torch.cat(sn_labels_list)
sn_preds = sn_logits.argmax(dim=-1)

sn_acc = (sn_preds == sn_labels).float().mean()
sorted_conf, sorted_idx = sn_conf.sort(descending=True)
sorted_errors = (sn_preds[sorted_idx] != sn_labels[sorted_idx]).float()
sn_aurc = aurc(sorted_conf, sorted_errors)

print(f"\nSelectiveNet: Acc={sn_acc:.2%}, AURC={sn_aurc:.4f}")
print(f"Mean selection prob = {sn_conf.mean():.4f}")

## Part 6: Method 4 — Self-Adaptive Training (SAT)

SAT improves calibration during training by blending hard labels with model
predictions:

$$y_{\text{adaptive}} = (1 - \alpha) \cdot y_{\text{hard}} + \alpha \cdot \text{softmax}(\text{logits})$$

This naturally produces better-calibrated confidence scores, which improves
selective prediction quality without an explicit rejection mechanism.

*Reference: Huang et al., "Self-Adaptive Training", NeurIPS 2020.*

In [ ]:
seed_everything(42)

sat_model = SelfAdaptiveTraining(
    ConvNet(num_classes=10, dropout_rate=0.0),
    num_classes=10,
    alpha_start=0.0,
    alpha_end=0.9,
    warmup_epochs=3,
).to(device)
optimizer = torch.optim.Adam(sat_model.parameters(), lr=0.001)

total_epochs = 10
print(f"Training SAT ({total_epochs} epochs, alpha: 0.0 -> 0.9, warmup=3)...")
sat_model.train()
for epoch in range(total_epochs):
    alpha = sat_model.get_alpha(epoch, total_epochs)
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = sat_model(inputs)
        loss = sat_model.sat_loss(logits, labels, alpha)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Acc={100.*correct/total:.2f}%, alpha={alpha:.3f}")

# Evaluate
sat_logits, sat_labels = collect_logits(sat_model, test_loader)
sat_preds = sat_logits.argmax(dim=-1)
sat_conf = F.softmax(sat_logits, dim=-1).max(dim=-1).values

sat_acc = (sat_preds == sat_labels).float().mean()
sorted_conf, sorted_idx = sat_conf.sort(descending=True)
sorted_errors = (sat_preds[sorted_idx] != sat_labels[sorted_idx]).float()
sat_aurc = aurc(sorted_conf, sorted_errors)

print(f"\nSAT: Acc={sat_acc:.2%}, AURC={sat_aurc:.4f}")

## Part 7: Method Comparison

Now let's compare all four methods using risk-coverage curves and AURC.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Left: Risk-coverage curves with bounds ---
ax = axes[0]
methods_data = {
    "SoftmaxThreshold": (test_logits, test_labels, msp_conf),
    "DeepGambler": (dg_class_logits, dg_labels, dg_conf),
    "SelectiveNet": (sn_logits, sn_labels, sn_conf),
    "SAT": (sat_logits, sat_labels, sat_conf),
}
colors = ["steelblue", "coral", "seagreen", "orchid"]

# Plot random and ideal bounds (use MSP's logits/labels — same test set)
overall_risk = 1.0 - (test_logits.argmax(-1) == test_labels).float().mean().item()
ax.axhline(overall_risk, color="gray", linestyle=":", linewidth=1.5, label="Random")

from incerto.sp.visual import _ideal_risk_curve
ideal_cov, ideal_risk = _ideal_risk_curve(test_logits, test_labels)
ax.plot(ideal_cov.numpy(), ideal_risk.numpy(), color="black", linestyle="--", linewidth=1.5, label="Ideal")

for (name, (logits, labels, conf)), color in zip(methods_data.items(), colors):
    cov, acc = accuracy_coverage_curve(logits, labels, conf)
    ax.plot(cov.numpy(), (1 - acc).numpy(), label=name, color=color, linewidth=2)

ax.set_xlabel("Coverage")
ax.set_ylabel("Risk (1 - accuracy)")
ax.set_title("Risk-Coverage Curves")
ax.legend(fontsize=8)
ax.grid(True, linestyle="--", linewidth=0.5)

# --- Center: Accuracy-coverage curves with bounds ---
ax = axes[1]

overall_acc = 1.0 - overall_risk
ax.axhline(overall_acc, color="gray", linestyle=":", linewidth=1.5, label="Random")

from incerto.sp.visual import _ideal_accuracy_curve
ideal_cov, ideal_acc = _ideal_accuracy_curve(test_logits, test_labels)
ax.plot(ideal_cov.numpy(), ideal_acc.numpy(), color="black", linestyle="--", linewidth=1.5, label="Ideal")

for (name, (logits, labels, conf)), color in zip(methods_data.items(), colors):
    cov, acc = accuracy_coverage_curve(logits, labels, conf)
    ax.plot(cov.numpy(), acc.numpy(), label=name, color=color, linewidth=2)

ax.set_xlabel("Coverage")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy-Coverage Curves")
ax.legend(fontsize=8)
ax.grid(True, linestyle="--", linewidth=0.5)

# --- Right: AURC bar chart ---
ax = axes[2]
methods = ["MSP", "DeepGambler", "SelectiveNet", "SAT"]
aurcs = [msp_aurc.item(), dg_aurc.item(), sn_aurc.item(), sat_aurc.item()]

bars = ax.bar(methods, aurcs, color=colors, alpha=0.85)
for bar, val in zip(bars, aurcs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel("AURC (lower is better)")
ax.set_title("AURC Comparison")
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
print("Method Comparison:")
print(f"{'Method':<20} {'Accuracy':>10} {'AURC':>10}")
print("-" * 42)
for name, acc, a in [
    ("SoftmaxThreshold", base_acc, msp_aurc),
    ("DeepGambler", dg_acc, dg_aurc),
    ("SelectiveNet", sn_acc, sn_aurc),
    ("SAT", sat_acc, sat_aurc),
]:
    print(f"{name:<20} {acc:>10.2%} {a:>10.4f}")

## Part 8: Metrics Deep-Dive

`incerto.sp` provides four metrics for evaluating selective predictors:

| Metric | Signature | Description |
|--------|-----------|-------------|
| `coverage` | `coverage(reject_mask)` | Fraction of accepted samples |
| `risk` | `risk(preds, labels, reject_mask)` | Error rate on accepted samples |
| `aurc` | `aurc(sorted_conf, sorted_errors)` | Area under risk-coverage curve |
| `accuracy_coverage_curve` | `accuracy_coverage_curve(logits, labels)` | Full accuracy-coverage curve |

In [ ]:
# Demonstrate metrics on the MSP method
threshold = 0.9
rejected = predictor.reject(msp_conf, threshold=threshold)

cov = coverage(rejected)
r = risk(base_preds, test_labels, rejected)

print(f"At threshold = {threshold}:")
print(f"  coverage(rejected)              = {cov:.4f}  ({cov:.1%} of samples predicted on)")
print(f"  risk(preds, labels, rejected)   = {r:.4f}  ({r:.2%} error rate on accepted)")
print(f"  selective accuracy              = {1-r:.2%}")
print()

# Full accuracy-coverage curve
cov_curve, acc_curve = accuracy_coverage_curve(test_logits, test_labels)
print(f"accuracy_coverage_curve returns {len(cov_curve)} points")
print(f"  Coverage range: [{cov_curve[0]:.4f}, {cov_curve[-1]:.4f}]")
print(f"  Accuracy range: [{acc_curve.min():.4f}, {acc_curve.max():.4f}]")

## Part 9: Visualizations

### Built-in Risk-Coverage and Accuracy-Coverage Plots

`plot_risk_coverage` and `plot_accuracy_coverage` generate publication-ready curves with
random and ideal (oracle) bounds. The **random** bound shows performance when rejecting
samples at random, while the **ideal** bound shows the best achievable performance with
a perfect confidence ranker.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Risk-coverage: MSP (default confidence = max softmax)
plot_risk_coverage(test_logits, test_labels, ax=axes[0, 0])
axes[0, 0].set_title("Risk-Coverage: SoftmaxThreshold")

# Risk-coverage: SAT (better calibrated)
plot_risk_coverage(sat_logits, sat_labels, ax=axes[0, 1])
axes[0, 1].set_title("Risk-Coverage: Self-Adaptive Training")

# Accuracy-coverage: MSP
plot_accuracy_coverage(test_logits, test_labels, ax=axes[1, 0])
axes[1, 0].set_title("Accuracy-Coverage: SoftmaxThreshold")

# Accuracy-coverage: SAT
plot_accuracy_coverage(sat_logits, sat_labels, ax=axes[1, 1])
axes[1, 1].set_title("Accuracy-Coverage: Self-Adaptive Training")

plt.tight_layout()
plt.show()

### Rejection Visualization

Let's visualize which Fashion-MNIST samples get rejected at a given threshold.

In [ ]:
# Visualize accepted vs rejected samples
test_batch, test_batch_labels = next(iter(test_loader))
test_batch_gpu = test_batch.to(device)

with torch.no_grad():
    batch_logits = model(test_batch_gpu)
    batch_conf = F.softmax(batch_logits, dim=-1).max(dim=-1).values.cpu()
    batch_preds = batch_logits.argmax(dim=-1).cpu()

threshold = 0.9
rejected = predictor.reject(batch_conf, threshold=threshold)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle(f"Selective Prediction (threshold={threshold})", fontsize=14, fontweight="bold")

for idx in range(16):
    ax = axes[idx // 8, idx % 8]
    img = test_batch[idx].squeeze().numpy()
    ax.imshow(img, cmap='gray')

    pred = batch_preds[idx].item()
    conf = batch_conf[idx].item()
    true_label = test_batch_labels[idx].item()
    is_rejected = rejected[idx].item()

    color = 'red' if is_rejected else ('green' if pred == true_label else 'orange')
    status = 'REJECT' if is_rejected else CLASS_NAMES[pred]
    ax.set_title(f"{status}\n{conf:.2f}", fontsize=8, color=color, fontweight="bold")
    ax.axis('off')

plt.tight_layout()
plt.show()

n_rejected = rejected[:16].sum().item()
print(f"Rejected {n_rejected}/16 displayed samples")
print(f"Total batch: {rejected.sum().item()}/{len(rejected)} rejected ({rejected.float().mean():.1%})")

## Part 10: Quick Factory API

`incerto.sp.make()` provides a convenient factory for creating selective predictors
by name.

In [ ]:
# Factory API: create methods by name
backbone = ConvNet(num_classes=10, dropout_rate=0.0)

msp = make("msp", backbone)
print(f"make('msp')          -> {type(msp).__name__}")

sat = make("sat", backbone, num_classes=10)
print(f"make('sat')          -> {type(sat).__name__}")

feat_backbone = ConvNet(num_classes=10, dropout_rate=0.0)
feat_backbone.fc2 = nn.Identity()

dg = make("gambler", feat_backbone, num_classes=10, num_features=128)
print(f"make('gambler')      -> {type(dg).__name__}")

feat_backbone2 = ConvNet(num_classes=10, dropout_rate=0.0)
feat_backbone2.fc2 = nn.Identity()

sn = make("selectivenet", feat_backbone2, num_classes=10, num_features=128)
print(f"make('selectivenet') -> {type(sn).__name__}")

## Summary

### Methods at a Glance

| Method | Approach | Pros | Cons |
|--------|----------|------|------|
| **SoftmaxThreshold** | Threshold max softmax | No training, simple | Uncalibrated confidence |
| **DeepGambler** | Learned abstain class | Learns when to abstain | Reward tuning |
| **SelectiveNet** | Dedicated selection head | Flexible coverage target | More parameters |
| **SAT** | Adaptive label smoothing | Better calibration | Indirect rejection signal |

### Key Takeaways

1. **SoftmaxThreshold** is the simplest starting point -- no extra training needed
2. **DeepGambler** and **SelectiveNet** learn to abstain end-to-end, often giving better risk-coverage tradeoffs
3. **SAT** improves calibration, which indirectly improves selective prediction quality
4. **AURC** is the primary metric -- lower means better separation of correct and incorrect predictions
5. Use `plot_risk_coverage()` and `plot_accuracy_coverage()` to visualize and compare methods — both include random and ideal (oracle) bounds for context

### Choosing a Method

- **Quick baseline:** `SoftmaxThreshold` (post-hoc, zero cost)
- **Best risk-coverage tradeoff:** `DeepGambler` or `SelectiveNet` (requires training)
- **Better calibration + rejection:** `SelfAdaptiveTraining`
- **Production:** Combine any method with calibration (see notebook 01)